In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:

from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': "AI Business Strategy, AI Hardware & Chips, AI in Action, AI Market Trends, Computer Vision, Featured News, Features, Infrastructure & Hardware, Inside AI, Manufacturing & Engineering AI, Opinion, Physical AI, Retail & Logistics AI\n\n### What LG and NVIDIA’s talks reveal about the future of physical AI\n\n## Latest\n\nAI Market Trends\n\nMay 29, 2026\n\n# Anthropic releases Claude Opus 4.8\n\nLogo on an Android smartphone as Google Pay is overhauling its payment infrastructure for an impending wave of transactions from AI agents.\n\nRetail & Logistics AI\n\nMay 28, 2026\n\n# Google Pay preps for AI agents with Universal Commerce Protocol\n\nNBA plans AI system for automatic out-of-bounds calls\n\nAI in Action\n\nMay 28, 2026\n\n# NBA plans AI system for automatic out-of-bounds calls [...] AI News is part of the TechForge Publications series\n\nTechForge\n\nAI apps including Claude on a mobile smartphone as Anthropic'

In [7]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [8]:
response = llm.invoke("Tell me joke about ai")
response

AIMessage(content='Why did the AI break up with its computer? \n\nBecause it found out it was seeing other programs on the side!', response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 12, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-f0fd700c-4391-402a-b51b-9ac6905ef282-0')

In [9]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

c:\Users\rohit\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [11]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool,get_weather_data]

In [12]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [13]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [16]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the captial of Inda"
        "and then find its current weather"
    )
})



> Entering new AgentExecutor chain...
I should first search for the capital of India and then use the weather data tool to find its current weather.
Action: tavily_search_results_json
Action Input: "capital of India"[{'url': 'https://www.countryreports.org/country/India/capital-city.htm', 'content': '|  |  |\n --- |\n| Capital City | New Delhi |\n| Capital location | 28 36 N, 77 12 E |\n| Capital - history | the city\'s name is associated with various myths and legends; the original name for the city may have been Dhilli or Dhillika; alternatively, the name could be a corruption of the Hindustani words "dehleez" or "dehali" - both terms meaning "threshold" or "gateway" - and indicative of the city as a gateway to the Gangetic Plain; after the British decided to move the capital of their Indian Empire from Calcutta to Delhi in 1911, they created a new governmental district south of the latter designated as New Delhi; the new capital was not formally inaugurated until 1931. |\n| Capita

In [17]:
print(response["output"])

The capital of India is New Delhi and its current weather is sunny with a temperature of 38°C and humidity at 25%.
